# Indic Parler Phonetic Evaluation
This notebook evaluates the Indic Parler-TTS model using the phonetically balanced Hindi dataset.
It generates audio for all phrases in `hindi_evaluation_set.json` and then uses Whisper ASR to compute WER and CER.

# **Indic Parler**

In [ ]:
!pip install git+https://github.com/huggingface/parler-tts.git soundfile transformers torch

  Cloning https://github.com/huggingface/parler-tts.git to /tmp/pip-req-build-rjn74ka2
  Running command git clone --filter=blob:none --quiet https://github.com/huggingface/parler-tts.git /tmp/pip-req-build-rjn74ka2
  Resolved https://github.com/huggingface/parler-tts.git to commit d108732cd57788ec86bc857d99a6cabd66663d68
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Cloning https://github.com/descriptinc/audiotools to /tmp/pip-install-6jj13w58/descript-audiotools_cd167189f3a04881b770fe33003092fd
  Running command git clone --filter=blob:none --quiet https://github.com/descriptinc/audiotools /tmp/pip-install-6jj13w58/descript-audiotools_cd167189f3a04881b770fe33003092fd
  Resolved https://github.com/descriptinc/audiotools to commit 348ebf2034ce24e2a91a553e3171cb00c0c71678
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.1/44.1 kB 3.0 MB/s eta 0:00:00
  Usin

In [ ]:
!pip install --upgrade protobuf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 327.1/327.1 kB 7.8 MB/s eta 0:00:00
  Attempting uninstall: protobuf
    Found existing installation: protobuf 4.25.3
    Uninstalling protobuf-4.25.3:
      Successfully uninstalled protobuf-4.25.3
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
descript-audiotools 0.7.4 requires protobuf!=4.24.0,<5.0.0,>=3.19.6, but you have protobuf 7.35.1 which is incompatible.
ydf 0.15.0 requires protobuf<7.0.0,>=5.29.1, but you have protobuf 7.35.1 which is incompatible.
opentelemetry-proto 1.38.0 requires protobuf<7.0,>=5.0, but you have protobuf 7.35.1 which is incompatible.
grpcio-status 1.71.2 requires protobuf<6.0dev,>=5.26.1, but you have protobuf 7.35.1 which is incompatible.
google-cloud-discoveryengine 0.13.12 requires protobuf!=4.21.0,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<7.0.0,>=3.20.2, but you have protobuf 7

In [ ]:
import os
os.kill(os.getpid(), 9)

In [ ]:
import os
# ==========================================
# THE BUG BYPASS
# Tell HuggingFace to completely ignore TensorFlow and JAX
# so it doesn't trigger the Protobuf crash.
os.environ["USE_TF"] = "0"
os.environ["USE_JAX"] = "0"
# ==========================================

import json
import torch
import soundfile as sf
from parler_tts import ParlerTTSForConditionalGeneration
from transformers import AutoTokenizer

print("Loading evaluation dataset...")
with open("hindi_evaluation_set.json", "r", encoding="utf-8") as f:
    eval_data = json.load(f)

# 1. Setup device and load the model
device = "cuda" if torch.cuda.is_available() else "cpu"
model_name = "ai4bharat/indic-parler-tts"

print(f"Loading {model_name} (this will download the weights)...")
model = ParlerTTSForConditionalGeneration.from_pretrained(model_name).to(device)

tokenizer = AutoTokenizer.from_pretrained(model_name)

# 2. Prepare output directory
output_dir = "indic_parler_hindi_eval"
os.makedirs(output_dir, exist_ok=True)

# 3. Define the Voice Prompts
voice_descriptions = {
    "Female": "A female speaker delivers a clear and expressive speech at a moderate pace. The recording is very high quality with no background noise.",
    "Male": "A male speaker delivers a clear and expressive speech at a moderate pace. The recording is very high quality with no background noise."
}

print("\nStarting Audio Generation...")

# 4. Generation Loop
for gender, description in voice_descriptions.items():
    print(f"\n--- Processing {gender} Voice ---")

    input_ids = tokenizer(description, return_tensors="pt").input_ids.to(device)

    for key, item in eval_data.items():
        text = item["text"]
        filename = f"{output_dir}/{item['id']}_{gender}.wav"

        print(f"Synthesizing {item['id']}...")

        prompt_input_ids = tokenizer(text, return_tensors="pt").input_ids.to(device)

        with torch.no_grad():
            generation = model.generate(
                input_ids=input_ids,
                prompt_input_ids=prompt_input_ids
            )

        audio_data = generation.cpu().numpy().squeeze()
        sample_rate = model.config.sampling_rate
        sf.write(filename, audio_data, sample_rate)

print(f"\n✅ All done! Evaluation audio files saved to '{output_dir}'.")

Loading evaluation dataset...
Loading ai4bharat/indic-parler-tts (this will download the weights)...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/3.75G [00:00<?, ?B/s]

  "_name_or_path": "google/flan-t5-large",
  "architectures": [
    "T5ForConditionalGeneration"
  ],
  "classifier_dropout": 0.0,
  "d_ff": 2816,
  "d_kv": 64,
  "d_model": 1024,
  "decoder_start_token_id": 0,
  "dense_act_fn": "gelu_new",
  "dropout_rate": 0.1,
  "eos_token_id": 1,
  "feed_forward_proj": "gated-gelu",
  "initializer_factor": 1.0,
  "is_encoder_decoder": true,
  "is_gated_act": true,
  "layer_norm_epsilon": 1e-06,
  "model_type": "t5",
  "n_positions": 512,
  "num_decoder_layers": 24,
  "num_heads": 16,
  "num_layers": 24,
  "output_past": true,
  "pad_token_id": 0,
  "relative_attention_max_distance": 128,
  "relative_attention_num_buckets": 32,
  "tie_word_embeddings": false,
  "transformers_version": "4.46.1",
  "use_cache": true,
  "vocab_size": 32128
}

  "_name_or_path": "ylacombe/dac_44khz",
  "architectures": [
    "DacModel"
  ],
  "codebook_dim": 8,
  "codebook_loss_weight": 1.0,
  "codebook_size": 1024,
  "commitment_loss_weight": 0.25,
  "decoder_hidden_si

...

tokenizer.model:   0%|          | 0.00/1.80M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/10.3M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/552 [00:00<?, ?B/s]

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.



Starting Audio Generation...

--- Processing Female Voice ---
Synthesizing HIN_01...
Synthesizing HIN_02...
Synthesizing HIN_03...
Synthesizing HIN_04...
Synthesizing HIN_05...
Synthesizing HIN_06...
Synthesizing HIN_07...
Synthesizing HIN_08...
Synthesizing HIN_09...
Synthesizing HIN_10...
Synthesizing HIN_11...
Synthesizing HIN_12...
Synthesizing HIN_13...
Synthesizing HIN_14...
Synthesizing HIN_15...
Synthesizing HIN_16...
Synthesizing HIN_17...
Synthesizing HIN_18...
Synthesizing HIN_19...
Synthesizing HIN_20...

--- Processing Male Voice ---
Synthesizing HIN_01...
Synthesizing HIN_02...
Synthesizing HIN_03...
Synthesizing HIN_04...
Synthesizing HIN_05...
Synthesizing HIN_06...
Synthesizing HIN_07...
Synthesizing HIN_08...
Synthesizing HIN_09...
Synthesizing HIN_10...
Synthesizing HIN_11...
Synthesizing HIN_12...
Synthesizing HIN_13...
Synthesizing HIN_14...
Synthesizing HIN_15...
Synthesizing HIN_16...
Synthesizing HIN_17...
Synthesizing HIN_18...
Synthesizing HIN_19...
Synthesiz

In [ ]:
import whisper
import jiwer
import json
import os
import pandas as pd
import warnings

warnings.filterwarnings("ignore")

print("Loading evaluation dataset...")
with open("hindi_evaluation_set.json", "r", encoding="utf-8") as f:
    eval_data = json.load(f)

print("Loading Whisper 'medium' model...")
model = whisper.load_model("medium")

output_dir = "indic_parler_hindi_eval"
results = []

print("\nStarting Transcription and Evaluation...\n")

# Evaluation Loop
for gender in ["Female", "Male"]:
    for key, item in eval_data.items():
        file_id = item["id"]
        ground_truth = item["text"]
        audio_path = f"{output_dir}/{file_id}_{gender}.wav"

        if not os.path.exists(audio_path):
            continue

        # Transcribe
        transcription_result = model.transcribe(audio_path, language="hi")
        hypothesis = transcription_result["text"].strip()

        # Calculate Error Rates
        try:
            wer = jiwer.wer(ground_truth, hypothesis)
            cer = jiwer.cer(ground_truth, hypothesis)
        except ValueError:
            wer = 1.0
            cer = 1.0

        # Store data
        results.append({
            "Test_ID": file_id,
            "Category": key,
            "Gender": gender,
            "WER": round(wer, 3),
            "CER": round(cer, 3),
            "Original_Text": ground_truth,
            "Whisper_Transcription": hypothesis
        })

        print(f"[{file_id} - {gender}] WER: {wer:.3f} | CER: {cer:.3f}")

# Create DataFrame and summarize
df_results = pd.DataFrame(results)

print("\n" + "="*40)
print("🏆 INDIC PARLER-TTS EVALUATION SUMMARY")
print("="*40)

summary = df_results.groupby("Gender")[["WER", "CER"]].mean()
print(summary)

# Save report
csv_filename = "indic_parler_whisper_evaluation.csv"
df_results.to_csv(csv_filename, index=False)
print(f"\n✅ Detailed breakdown saved to '{csv_filename}'.")

Loading evaluation dataset...
Loading Whisper 'medium' model...

Starting Transcription and Evaluation...

[HIN_01 - Female] WER: 0.733 | CER: 0.333
[HIN_02 - Female] WER: 0.556 | CER: 0.182
[HIN_03 - Female] WER: 0.750 | CER: 0.270
[HIN_04 - Female] WER: 0.545 | CER: 0.245
[HIN_05 - Female] WER: 1.000 | CER: 0.768
[HIN_06 - Female] WER: 0.833 | CER: 0.455
[HIN_07 - Female] WER: 0.833 | CER: 0.500
[HIN_08 - Female] WER: 1.071 | CER: 0.820
[HIN_09 - Female] WER: 0.929 | CER: 0.591
[HIN_10 - Female] WER: 0.857 | CER: 0.671
[HIN_11 - Female] WER: 0.786 | CER: 0.437
[HIN_12 - Female] WER: 0.769 | CER: 0.433
[HIN_13 - Female] WER: 0.846 | CER: 0.509
[HIN_14 - Female] WER: 1.000 | CER: 0.828
[HIN_15 - Female] WER: 1.133 | CER: 0.713
[HIN_16 - Female] WER: 0.765 | CER: 0.380
[HIN_17 - Female] WER: 0.857 | CER: 0.592
[HIN_18 - Female] WER: 0.857 | CER: 0.437
[HIN_19 - Female] WER: 0.800 | CER: 0.558
[HIN_20 - Female] WER: 0.533 | CER: 0.209
[HIN_01 - Male] WER: 1.000 | CER: 0.950
[HIN_02 - Mal

In [ ]:
import shutil
from google.colab import files

print("Zipping the audio files...")
# Compress the folder into a file named 'xtts_audio_results.zip'
shutil.make_archive('indic_parler_audio_results', 'zip', 'indic_parler_hindi_eval')

print("Starting download...")
# Trigger the browser download
files.download('indic_parler_audio_results.zip')

Zipping the audio files...
Starting download...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>